In [1]:
# Define the target configuration (Goal State)
# 0 represents the empty/blank space
GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

def find_blank(state):
    """Finds the (row, col) coordinates of the blank tile (0)."""
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c

def get_neighbors(state):
    """Generates all valid next states by shifting the blank tile."""
    neighbors = []
    r, c = find_blank(state)

    moves = [
        (-1, 0, 'Up'),
        (1, 0, 'Down'),
        (0, -1, 'Left'),
        (0, 1, 'Right')
    ]

    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = [list(row) for row in state]
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            neighbors.append((tuple(tuple(row) for row in new_state), action))

    return neighbors

def depth_limited_dfs(state, path, depth_limit, visited_in_path):
    """
    Performs a depth-limited DFS tracking states strictly along the current path branch
    to avoid local backtracking cycles.
    """
    if state == GOAL_STATE:
        return path, False  # Found solution, no cutoff occurred

    if depth_limit == 0:
        return None, True   # Cutoff hit (there are deeper paths to explore)

    cutoff_occurred = False
    visited_in_path.add(state)

    for neighbor_state, action in get_neighbors(state):
        if neighbor_state not in visited_in_path:
            # Recurse with depth limit reduced by 1
            result, remaining_cutoff = depth_limited_dfs(
                neighbor_state, path + [action], depth_limit - 1, visited_in_path
            )

            if result is not None:
                return result, False
            if remaining_cutoff:
                cutoff_occurred = True

    visited_in_path.remove(state)  # Backtrack
    return None, cutoff_occurred

def solve_8_puzzle_ids(start_state, max_depth=50):
    """Solves the 8-puzzle using Iterative Deepening Search (IDS)."""
    for depth in range(max_depth):
        print(f"Searching at depth limit: {depth}")
        visited_in_path = set()

        solution, cutoff = depth_limited_dfs(start_state, [], depth, visited_in_path)

        if solution is not None:
            return solution
        if not cutoff:
            # Explored the complete game graph up to dead ends and found nothing
            break

    return None

def print_board(state):
    """Helper function to print the 3x3 matrix beautifully."""
    for row in state:
        print(" ".join(str(tile) if tile != 0 else "_" for tile in row))
    print()

# =====================================================================
# Example Execution
# =====================================================================
if __name__ == "__main__":
    # Define a starting board layout (Solvable in 5 moves)
    initial_state = (
        (1, 2, 3),
        (0, 4, 6),
        (7, 5, 8)
    )

    print("--- Initial State ---")
    print_board(initial_state)

    solution_path = solve_8_puzzle_ids(initial_state)

    if solution_path:
        print(f"\nSolution found in {len(solution_path)} moves!")
        print(f"Sequence of actions: {' -> '.join(solution_path)}")
    else:
        print("\nNo solution found within depth limits.")


--- Initial State ---
1 2 3
_ 4 6
7 5 8

Searching at depth limit: 0
Searching at depth limit: 1
Searching at depth limit: 2
Searching at depth limit: 3

Solution found in 3 moves!
Sequence of actions: Right -> Down -> Right
